In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pygris import tracts 
from pygris.data import get_lodes
import matplotlib.pyplot as plt


In [2]:
NY_2009_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2009.csv")
NY_2010_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2010.csv")
NY_2011_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2011.csv")
NY_2012_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2012.csv")
NY_2013_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2013.csv")
NY_2014_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2014.csv")
NY_2015_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2015.csv")
NY_2016_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2016.csv")
NY_2017_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2017.csv")
NY_2018_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2018.csv")
NY_2019_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2019.csv")
NY_2020_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2020.csv")

In [3]:
def get_hudson_commuters(commute_df, tracts_hudson):
    """
    Filters LODES data for workers commuting to any of the specified Hudson tracts.

    Returns:
        DataFrame with columns: h_trct, w_trct, hudson_workers
    """
    df = commute_df.assign(
        w_trct=commute_df["w_geocode"].astype(str).str.slice(stop=11),
        h_trct=commute_df["h_geocode"].astype(str).str.slice(stop=11)
    )

    return (
        df.query('w_trct in @tracts_hudson')
        .groupby(["h_trct", "w_trct"], as_index=False)
        .agg({"S000": "sum"})
        .rename(columns={"S000": "hudson_workers"})
    )


In [4]:
def get_total_workers(commute_df):
    """
    Aggregates total workers per home tract.
    """
    return (
        commute_df
        .assign(h_trct=commute_df["h_geocode"].astype(str).str.slice(stop=11))
        .groupby("h_trct", as_index=False)
        .agg({"S000": "sum"})
        .rename(columns={"S000": "total_workers"})
    )


In [5]:
def compute_hudson_share(hudson_df, total_df):
    """
    Merges hudson-bound commuter counts with total and computes commuters per 1000.
    """
    return (
        hudson_df
        .merge(total_df, on="h_trct", how="left")
        .assign(hudson_workers_per_1000=lambda df: 1000 * df["hudson_workers"] / df["total_workers"])
    )


In [6]:
def attach_geometries_and_distance(commute_stats_df, NY_tracts, crs_epsg=2263):
    """
    Adds geometry for home and work tracts and computes distance in kilometers.
    """
    # Merge home tract geometry
    home_geom = NY_tracts.merge(commute_stats_df, left_on="GEOID", right_on="h_trct", how="left")

    # Merge work tract geometry
    w_geom = NY_tracts[["GEOID", "geometry"]].rename(columns={"GEOID": "w_trct", "geometry": "w_geometry"})
    merged = home_geom.merge(w_geom, on="w_trct", how="left")

    # Reproject and compute distance
    merged = merged.to_crs(epsg=crs_epsg)
    merged["w_geometry"] = merged["w_geometry"].to_crs(epsg=crs_epsg)
    merged["tract_distance"] = merged.geometry.distance(merged["w_geometry"])
    merged["distance_km"] = (merged["tract_distance"] * 0.0003048).round(2)

    return merged


In [7]:
NY_tracts = tracts(state = "NY", cb = True, 
                   county=['Bronx County', 'Kings County', 'New York County', 'Queens County', 'Richmond County'
],
                    year = 2020, cache = True)

Using FIPS code '36' for input 'NY'
Using FIPS code '005' for input 'Bronx County'
Using FIPS code '047' for input 'Kings County'
Using FIPS code '061' for input 'New York County'
Using FIPS code '081' for input 'Queens County'
Using FIPS code '085' for input 'Richmond County'


In [8]:
tracts_hudson = ['36061009903', '36061009902', '36061009901', '36061009900']

# 2009
hudson_2009 = get_hudson_commuters(NY_2009_commute_df, tracts_hudson)
total_2009 = get_total_workers(NY_2009_commute_df)
hudson_2009_stats = compute_hudson_share(hudson_2009, total_2009)
gdf_2009 = attach_geometries_and_distance(hudson_2009_stats, NY_tracts)

# 2010
hudson_2010 = get_hudson_commuters(NY_2010_commute_df, tracts_hudson)
total_2010 = get_total_workers(NY_2010_commute_df)
hudson_2010_stats = compute_hudson_share(hudson_2010, total_2010)
gdf_2010 = attach_geometries_and_distance(hudson_2010_stats, NY_tracts)

# 2011
hudson_2011 = get_hudson_commuters(NY_2011_commute_df, tracts_hudson)
total_2011 = get_total_workers(NY_2011_commute_df)
hudson_2011_stats = compute_hudson_share(hudson_2011, total_2011)
gdf_2011 = attach_geometries_and_distance(hudson_2011_stats, NY_tracts)

# 2012
hudson_2012 = get_hudson_commuters(NY_2012_commute_df, tracts_hudson)
total_2012 = get_total_workers(NY_2012_commute_df)
hudson_2012_stats = compute_hudson_share(hudson_2012, total_2012)
gdf_2012 = attach_geometries_and_distance(hudson_2012_stats, NY_tracts)

# 2013
hudson_2013 = get_hudson_commuters(NY_2013_commute_df, tracts_hudson)
total_2013 = get_total_workers(NY_2013_commute_df)
hudson_2013_stats = compute_hudson_share(hudson_2013, total_2013)
gdf_2013 = attach_geometries_and_distance(hudson_2013_stats, NY_tracts)

# 2014
hudson_2014 = get_hudson_commuters(NY_2014_commute_df, tracts_hudson)
total_2014 = get_total_workers(NY_2014_commute_df)
hudson_2014_stats = compute_hudson_share(hudson_2014, total_2014)
gdf_2014 = attach_geometries_and_distance(hudson_2014_stats, NY_tracts)

# 2015
hudson_2015 = get_hudson_commuters(NY_2015_commute_df, tracts_hudson)
total_2015 = get_total_workers(NY_2015_commute_df)
hudson_2015_stats = compute_hudson_share(hudson_2015, total_2015)
gdf_2015 = attach_geometries_and_distance(hudson_2015_stats, NY_tracts)

# 2016
hudson_2016 = get_hudson_commuters(NY_2016_commute_df, tracts_hudson)
total_2016 = get_total_workers(NY_2016_commute_df)
hudson_2016_stats = compute_hudson_share(hudson_2016, total_2016)
gdf_2016 = attach_geometries_and_distance(hudson_2016_stats, NY_tracts)

# 2017
hudson_2017 = get_hudson_commuters(NY_2017_commute_df, tracts_hudson)
total_2017 = get_total_workers(NY_2017_commute_df)
hudson_2017_stats = compute_hudson_share(hudson_2017, total_2017)
gdf_2017 = attach_geometries_and_distance(hudson_2017_stats, NY_tracts)

# 2018
hudson_2018 = get_hudson_commuters(NY_2018_commute_df, tracts_hudson)
total_2018 = get_total_workers(NY_2018_commute_df)
hudson_2018_stats = compute_hudson_share(hudson_2018, total_2018)
gdf_2018 = attach_geometries_and_distance(hudson_2018_stats, NY_tracts)

# 2019
hudson_2019 = get_hudson_commuters(NY_2019_commute_df, tracts_hudson)
total_2019 = get_total_workers(NY_2019_commute_df)
hudson_2019_stats = compute_hudson_share(hudson_2019, total_2019)
gdf_2019 = attach_geometries_and_distance(hudson_2019_stats, NY_tracts)

# 2020
hudson_2020 = get_hudson_commuters(NY_2020_commute_df, tracts_hudson)
total_2020 = get_total_workers(NY_2020_commute_df)
hudson_2020_stats = compute_hudson_share(hudson_2020, total_2020)
gdf_2020 = attach_geometries_and_distance(hudson_2020_stats, NY_tracts)


In [ ]:
bin_order = ["<1 km", "1–3 km", "3–5 km", "5–7 km", "7–10 km", "10+ km"]
x = np.arange(len(bin_order))

# Set up plot
fig, ax = plt.subplots(figsize=(10, 6))

# Color mapping for periods
color_map = {
    "Before 2015": "#1f77b4",  # blue
    "After 2015": "#d62728"    # red
}

for period in ["Before 2015", "After 2015"]:
    df = grouped[grouped["period"] == period].set_index("distance_bin_custom").reindex(bin_order)

    y = df["avg"].values.astype(float)
    ci = df["ci95"].values.astype(float)

    # Plot mean line
    ax.plot(x, y, label=period, marker='o', color=color_map[period])

    # Plot 95% CI band
    ax.fill_between(x, y - ci, y + ci, color=color_map[period], alpha=0.2, label=f"{period} 95% CI")

# Formatting
ax.set_xticks(x)
ax.set_xticklabels(bin_order)
ax.set_xlabel("Distance Bin (km)")
ax.set_ylabel("Avg. Commuters per 1000")
ax.set_title("Hudson Commuters per 1000 with 95% Confidence Intervals")
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
ax.set_ylim(0, grouped["avg"].max() + 2)

plt.tight_layout()
plt.show()


In [ ]:
 #Create 5 km bins and labels
bin_edges = range(0, int(max(gdf_2014["distance_km"].max(), gdf_2016["distance_km"].max())) + 10, 5)
bin_labels = [f"{i}-{i+5} km" for i in bin_edges[:-1]]


# Bin both datasets
gdf_2014["distance_bin_5km"] = pd.cut(gdf_2014["distance_km"], bins=bin_edges, labels=bin_labels)
gdf_2016["distance_bin_5km"] = pd.cut(gdf_2016["distance_km"], bins=bin_edges, labels=bin_labels)
#gdf_2015["distance_bin_5km"] = pd.cut(gdf_2016["distance_km"], bins=bin_edges, labels=bin_labels)


# Function to compute summary with 95% CI
def compute_summary_with_ci(df, label):
    grouped = (
        df.groupby("distance_bin_5km")["hudson_workers_per_1000"]
        .agg(['mean', 'std', 'count'])
        .reset_index()
        .rename(columns={'mean': 'avg', 'std': 'stddev', 'count': 'n'})
    )
    grouped['ci95'] = 1.96 * grouped['stddev'] / np.sqrt(grouped['n'])
    grouped['year'] = label
    return grouped

# Compute summaries
summary_2014 = compute_summary_with_ci(gdf_2014, "2014")
#summary_2015 = compute_summary_with_ci(gdf_2015, "2015")
summary_2016 = compute_summary_with_ci(gdf_2016, "2016")

# Create shaded confidence bands
fig = go.Figure()

for summary_df, color, fill in zip(
    [summary_2014,  summary_2016],
    ["blue", "orange", "green"],
    ["rgba(0, 0, 255, 0.2)", "rgba(255, 165, 0, 0.2)", "rgba(0, 128, 0, 0.2)"]
):
    label = summary_df["year"].iloc[0]

    # Line and markers
    fig.add_trace(go.Scatter(
        x=summary_df["distance_bin_5km"],
        y=summary_df["avg"],
        mode="lines+markers",
        name=label,
        line=dict(color=color)
    ))

    # Upper bound (invisible line)
    fig.add_trace(go.Scatter(
        x=summary_df["distance_bin_5km"],
        y=summary_df["avg"] + summary_df["ci95"],
        mode="lines",
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip'
    ))

    # Lower bound (fills to upper line)
    fig.add_trace(go.Scatter(
        x=summary_df["distance_bin_5km"],
        y=summary_df["avg"] - summary_df["ci95"],
        mode="lines",
        line=dict(width=0),
        fill='tonexty',
        fillcolor=fill,
        hoverinfo='skip',
        showlegend=False
    ))


# Layout
fig.update_layout(
    title="Average Hudson Commuters per 5km Distance Bin (with 95% CI)",
    xaxis_title="Distance Bin (km)",
    yaxis_title="Avg. Commuters per 1000",
    template="plotly_white",
    yaxis=dict(range=[0, 3.5])
)

fig.show()

C:\Users\fiodo\AppData\Local\Temp\ipykernel_17820\1177515277.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("distance_bin_5km")["hudson_workers_per_1000"]
C:\Users\fiodo\AppData\Local\Temp\ipykernel_17820\1177515277.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("distance_bin_5km")["hudson_workers_per_1000"]


In [33]:
# Define custom bins and labels
custom_edges = [0, 1, 3, 5, 7, 10, float("inf")]
custom_labels = ["<1 km", "1–3 km", "3–5 km", "5–7 km", "7–10 km", "10+ km"]

# Apply binning to both datasets
gdf_2014["distance_bin_custom"] = pd.cut(
    gdf_2014["distance_km"],
    bins=custom_edges,
    labels=custom_labels,
    right=False  # optional: includes the left edge, excludes the right
)

gdf_2016["distance_bin_custom"] = pd.cut(
    gdf_2016["distance_km"],
    bins=custom_edges,
    labels=custom_labels,
    right=False
)
# Function to compute summary with 95% CI
def compute_summary_with_ci(df, label):
    grouped = (
        df.groupby("distance_bin_custom")["hudson_workers_per_1000"]
        .agg(['mean', 'std', 'count'])
        .reset_index()
        .rename(columns={'mean': 'avg', 'std': 'stddev', 'count': 'n'})
    )
    grouped['ci95'] = 1.96 * grouped['stddev'] / np.sqrt(grouped['n'])
    grouped['year'] = label
    return grouped

# Compute summaries
summary_2014 = compute_summary_with_ci(gdf_2014, "2014")
#summary_2015 = compute_summary_with_ci(gdf_2015, "2015")
summary_2016 = compute_summary_with_ci(gdf_2016, "2016")

# Create shaded confidence bands
fig = go.Figure()

for summary_df, color, fill in zip(
    [summary_2014,  summary_2016],
    ["blue", "orange", "green"],
    ["rgba(0, 0, 255, 0.2)", "rgba(255, 165, 0, 0.2)", "rgba(0, 128, 0, 0.2)"]
):
    label = summary_df["year"].iloc[0]

    # Line and markers
    fig.add_trace(go.Scatter(
        x=summary_df["distance_bin_custom"],
        y=summary_df["avg"],
        mode="lines+markers",
        name=label,
        line=dict(color=color)
    ))

    # Upper bound (invisible line)
    fig.add_trace(go.Scatter(
        x=summary_df["distance_bin_custom"],
        y=summary_df["avg"] + summary_df["ci95"],
        mode="lines",
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip'
    ))

    # Lower bound (fills to upper line)
    fig.add_trace(go.Scatter(
        x=summary_df["distance_bin_custom"],
        y=summary_df["avg"] - summary_df["ci95"],
        mode="lines",
        line=dict(width=0),
        fill='tonexty',
        fillcolor=fill,
        hoverinfo='skip',
        showlegend=False
    ))


# Layout
fig.update_layout(
    title="Average Hudson Commuters per 5km Distance Bin (with 95% CI)",
    xaxis_title="Distance Bin (km)",
    yaxis_title="Avg. Commuters per 1000",
    template="plotly_white",
    yaxis=dict(range=[0, 3.5])
)

fig.show()

C:\Users\fiodo\AppData\Local\Temp\ipykernel_17820\2457797790.py:22: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\fiodo\AppData\Local\Temp\ipykernel_17820\2457797790.py:22: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [36]:
# Step 1: Concatenate all gdf_YEARs
gdf_all_years = pd.concat(
    [gdf_2009.assign(year=2009),
     gdf_2010.assign(year=2010),
     gdf_2011.assign(year=2011),
     gdf_2012.assign(year=2012),
     gdf_2013.assign(year=2013),
     gdf_2014.assign(year=2014),
     gdf_2015.assign(year=2015),
     gdf_2016.assign(year=2016),
     gdf_2017.assign(year=2017),
     gdf_2018.assign(year=2018),
     gdf_2019.assign(year=2019),
     gdf_2020.assign(year=2020)],
    ignore_index=True
)

# Step 2: Add period column
gdf_all_years["period"] = gdf_all_years["year"].apply(lambda y: "Before 2015" if y < 2015 else "After 2015")

# Step 3: Bin distances
bin_edges = range(0, int(gdf_all_years["distance_km"].max()) + 10, 5)
bin_labels = [f"{i}-{i+5} km" for i in bin_edges[:-1]]
gdf_all_years["distance_bin_5km"] = pd.cut(gdf_all_years["distance_km"], bins=bin_edges, labels=bin_labels)
custom_edges = [0, 1, 3, 5, 7, 10, float("inf")]
custom_labels = ["<1 km", "1–3 km", "3–5 km", "5–7 km", "7–10 km", ">10 km"]

# Apply binning
gdf_all_years["distance_bin_custom"] = pd.cut(
    gdf_all_years["distance_km"],
    bins=custom_edges,
    labels=custom_labels,
    right=False  # This ensures intervals like [0,1), [1,3), ...
)


In [26]:
gdf_all_years

,STATEFP,COUNTYFP,TRACTCE,AFFGEOID,GEOID,NAME,NAMELSAD,STUSPS,NAMELSADCO,STATE_NAME,...,w_trct,hudson_workers,total_workers,hudson_workers_per_1000,w_geometry,tract_distance,distance_km,year,distance_bin_5km,period
0,36,005,009200,1400000US36005009200,36005009200,92,Census Tract 92,NY,Bronx County,New York,...,36061009901,1.0,1685.0,0.593472,"POLYGON ((979813.044 212302.264, 980090.428 21...",49821.520200,15.19,2009,15-20 km,Before 2015
1,36,005,009200,1400000US36005009200,36005009200,92,Census Tract 92,NY,Bronx County,New York,...,36061009902,1.0,1685.0,0.593472,"POLYGON ((980302.975 214947.112, 982253.83 214...",48877.636675,14.90,2009,10-15 km,Before 2015
2,36,005,009200,1400000US36005009200,36005009200,92,Census Tract 92,NY,Bronx County,New York,...,36061009903,2.0,1685.0,1.186944,"POLYGON ((980363.385 215273.272, 980661.843 21...",46961.811634,14.31,2009,10-15 km,Before 2015
3,36,005,006400,1400000US36005006400,36005006400,64,Census Tract 64,NY,Bronx County,New York,...,NaN,NaN,NaN,NaN,None,NaN,NaN,2009,NaN,Before 2015
4,36,005,036501,1400000US36005036501,36005036501,365.01,Census Tract 365.01,NY,Bronx County,New York,...,36061009902,3.0,1078.0,2.782931,"POLYGON ((980302.975 214947.112, 982253.83 214...",45739.451965,13.94,2009,10-15 km,Before 2015
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59575,36,081,141700,1400000US36081141700,36081141700,1417,Census Tract 1417,NY,Queens County,New York,...,36061009901,2.0,1484.0,1.347709,"POLYGON ((979813.044 212302.264, 980090.428 21...",56936.135497,17.35,2020,15-20 km,After 2015
59576,36,081,141700,1400000US36081141700,36081141700,1417,Census Tract 1417,NY,Queens County,New York,...,36061009902,1.0,1484.0,0.673854,"POLYGON ((980302.975 214947.112, 982253.83 214...",56441.630623,17.20,2020,15-20 km,After 2015
59577,36,081,141700,1400000US36081141700,36081141700,1417,Census Tract 1417,NY,Queens County,New York,...,36061009903,3.0,1484.0,2.021563,"POLYGON ((980363.385 215273.272, 980661.843 21...",55485.392270,16.91,2020,15-20 km,After 2015
59578,36,085,014608,1400000US36085014608,36085014608,146.08,Census Tract 146.08,NY,Richmond County,New York,...,36061009902,4.0,1120.0,3.571429,"POLYGON ((980302.975 214947.112, 982253.83 214...",79404.928932,24.20,2020,20-25 km,After 2015


In [27]:
# Step 1: Collect all gdf_YEARs into one DataFrame with a year column
grouped = (
    gdf_all_years
    .groupby(["period", "distance_bin_5km"])["hudson_workers_per_1000"]
    .agg(['mean', 'std', 'count'])
    .reset_index()
    .rename(columns={'mean': 'avg', 'std': 'stddev', 'count': 'n'})
)

# Calculate 95% CI (Z = 1.96 for normal distribution)
grouped['ci95'] = 1.96 * grouped['stddev'] / np.sqrt(grouped['n'])

# Step 5: Plot with CI as bands
fig = go.Figure()

color_map = {
    "Before 2015": "#1f77b4",  # Blue
    "After 2015": "#2ca02c"    # Green
}

for period in ["Before 2015", "After 2015"]:
    df = grouped[grouped["period"] == period]
    
    # Main line
    fig.add_trace(go.Scatter(
        x=df["distance_bin_5km"],
        y=df["avg"],
        mode="lines+markers",
        name=period,
        line=dict(width=2)
    ))
    
    fig.add_trace(go.Scatter(
        x=df["distance_bin_5km"],
        y=df["avg"],
        mode="lines+markers",
        name=period,
        error_y=dict(
            type="data",
            array=df["ci95"],
            thickness=1.5,
            width=5,  # width of the cap
            color="gray"
        )
        ,showlegend=False
        ))

# Layout
fig.update_layout(
    title="Hudson Commuters per 1000 with 95% Confidence Intervals",
    xaxis_title="Distance Bin (km)",
    yaxis_title="Avg. Commuters per 1000",
    template="plotly_white",
    yaxis=dict(range=[0, 3.5]),
    legend_title="Period"
)

fig.show()

C:\Users\fiodo\AppData\Local\Temp\ipykernel_17820\3165876044.py:4: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [28]:
# Step 4 (updated): Group and compute mean, std, count
grouped = (
    gdf_all_years
    .groupby(["period", "distance_bin_5km"])["hudson_workers_per_1000"]
    .agg(['mean', 'std', 'count'])
    .reset_index()
    .rename(columns={'mean': 'avg', 'std': 'stddev', 'count': 'n'})
)

# Calculate 95% CI (Z = 1.96 for normal distribution)
grouped['ci95'] = 1.96 * grouped['stddev'] / np.sqrt(grouped['n'])

# Step 5: Plot with CI as bands
fig = go.Figure()

for period in ["Before 2015", "After 2015"]:
    df = grouped[grouped["period"] == period]
    
    # Main line
    fig.add_trace(go.Scatter(
        x=df["distance_bin_5km"],
        y=df["avg"],
        mode="lines+markers",
        name=period,
        line=dict(width=2)
    ))
    
    # Upper bound
    fig.add_trace(go.Scatter(
        x=df["distance_bin_5km"],
        y=df["avg"] + df["ci95"],
        mode="lines",
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip'
    ))

    # Lower bound
    fig.add_trace(go.Scatter(
        x=df["distance_bin_5km"],
        y=df["avg"] - df["ci95"],
        mode="lines",
        line=dict(width=0),
        fill='tonexty',
        fillcolor='rgba(0,100,80,0.2)' if period == "Before 2015" else 'rgba(200,100,100,0.2)',
        name=f"{period} 95% CI",
        hoverinfo='skip'
    ))

# Layout
fig.update_layout(
    title="Hudson Commuters per 1000 with 95% Confidence Intervals",
    xaxis_title="Distance Bin (km)",
    yaxis_title="Avg. Commuters per 1000",
    template="plotly_white",
    yaxis=dict(range=[0, 3.5]),
    legend_title="Period"
)

fig.show()

C:\Users\fiodo\AppData\Local\Temp\ipykernel_17820\3007223037.py:4: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [37]:
# Step 4 (updated): Group and compute mean, std, count
grouped = (
    gdf_all_years
    .groupby(["period", "distance_bin_custom"])["hudson_workers_per_1000"]
    .agg(['mean', 'std', 'count'])
    .reset_index()
    .rename(columns={'mean': 'avg', 'std': 'stddev', 'count': 'n'})
)

# Calculate 95% CI (Z = 1.96 for normal distribution)
grouped['ci95'] = 1.96 * grouped['stddev'] / np.sqrt(grouped['n'])

# Step 5: Plot with CI as bands
fig = go.Figure()

for period in ["Before 2015", "After 2015"]:
    df = grouped[grouped["period"] == period]
    
    # Main line
    fig.add_trace(go.Scatter(
        x=df["distance_bin_custom"],
        y=df["avg"],
        mode="lines+markers",
        name=period,
        line=dict(width=2)
    ))
    
    # Upper bound
    fig.add_trace(go.Scatter(
        x=df["distance_bin_custom"],
        y=df["avg"] + df["ci95"],
        mode="lines",
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip'
    ))

    # Lower bound
    fig.add_trace(go.Scatter(
        x=df["distance_bin_custom"],
        y=df["avg"] - df["ci95"],
        mode="lines",
        line=dict(width=0),
        fill='tonexty',
        fillcolor='rgba(0,100,80,0.2)' if period == "Before 2015" else 'rgba(200,100,100,0.2)',
        name=f"{period} 95% CI",
        hoverinfo='skip'
    ))

# Layout
fig.update_layout(
    title="Hudson Commuters per 1000 with 95% Confidence Intervals",
    xaxis_title="Distance Bin (km)",
    yaxis_title="Avg. Commuters per 1000",
    template="plotly_white",
    yaxis=dict(range=[0, 3.5]),
    legend_title="Period"
)

fig.show()

C:\Users\fiodo\AppData\Local\Temp\ipykernel_17820\2838496826.py:4: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

